# Test 3: Water Dimer O-O Potential Energy Scan

Colab-runnable driver. Session-safe: re-running a scan cell after a Colab
disconnect resumes from the last checkpointed point rather than starting
over.

**Two-environment split, read before running anything:** `mace-torch`
hard-pins `e3nn==0.4.4`; `fairchem-core` (needed for UMA-S) requires
`e3nn>=0.5`. These genuinely conflict -- they cannot be pip-installed into
the same environment. So this notebook has two independent parts:

- **Part A** (below): ANI-2x + MACE-OFF23-small.
- **Part B**: UMA-S.

Run Part A, download its results, then **Runtime > Restart session**
before running Part B. Do not run both parts in the same runtime without
restarting in between -- installing one stack on top of the other will
break one of them.

`mace-off23-small` (Part A) is the acceptance-critical model for this
session: it must show a spurious minimum below ~1.0 A that is deeper than
the physical minimum near 2.9 A (the published MACE-OFF23 failure mode).

## Part A: ANI-2x + MACE-OFF23-small

### A1. Clone the repo and install the `ani-mace` stack

In [ ]:
# Edit REPO_URL if this notebook was opened outside the cloned repo.
import os

REPO_URL = "<your-fork-or-repo-url>"
REPO_DIR = "/content/mlip-audit"

if not os.path.isdir(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
%cd $REPO_DIR

In [ ]:
!bash setup.sh ani-mace

### A2. Run Test 3 (resumable -- safe to re-run after a disconnect)

In [ ]:
!python -m mlip_audit.test3_dimer --model mace-off23-small --verbose

In [ ]:
!python -m mlip_audit.test3_dimer --model ani2x --verbose

### A3. Plot and check for the spurious minimum

In [ ]:
from mlip_audit.plotting import plot_dimer_scan, check_spurious_minimum
from mlip_audit.config import RESULTS_DIR

csv_dir = RESULTS_DIR / "test3_dimer"
csv_paths = sorted(csv_dir.glob("*.csv"))
print("Found:", csv_paths)

out_path = plot_dimer_scan(csv_paths, show=True)
print("Saved:", out_path)

for p in csv_paths:
    result = check_spurious_minimum(p)
    print(f"\n{p.stem}")
    for k, v in result.items():
        print(f"  {k}: {v}")

### A4. Download results before restarting the runtime

Colab sessions/runtimes are ephemeral -- pull results down now, before
restarting for Part B.

In [ ]:
from google.colab import files
import shutil

archive_path = shutil.make_archive("/content/test3_dimer_results_partA", "zip", str(RESULTS_DIR))
files.download(archive_path)

---
## Part B: UMA-S

**Before continuing: Runtime > Restart session** (do this now if you just
ran Part A in this runtime).

### B1. Re-clone (if needed) and install the `uma` stack

In [ ]:
import os

REPO_URL = "<your-fork-or-repo-url>"
REPO_DIR = "/content/mlip-audit"

if not os.path.isdir(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
%cd $REPO_DIR

In [ ]:
!bash setup.sh uma

### B2. HuggingFace login (only if setup.sh reported no cached token)

This opens an interactive prompt -- your token is entered directly into the
widget, never typed into a cell, so it won't end up in notebook output or
git history. Make sure you've accepted the gated UMA model license on
huggingface.co with this account before continuing.

In [ ]:
from huggingface_hub import get_token

if get_token() is None:
    from huggingface_hub import login
    login()  # interactive widget; do not pass token= here
else:
    print("Already logged in.")

### B3. Sanity check: charge/spin actually reaches UMA-S

In [ ]:
!python -m pytest tests/test_charge_spin.py -v

### B4. Run Test 3 for UMA-S (resumable)

In [ ]:
!python -m mlip_audit.test3_dimer --model uma-s-1p1 --verbose

### B5. Plot (UMA-S alone, or overlaid with Part A's CSVs if you copied them back in)

In [ ]:
from mlip_audit.plotting import plot_dimer_scan, check_spurious_minimum
from mlip_audit.config import RESULTS_DIR

csv_dir = RESULTS_DIR / "test3_dimer"
csv_paths = sorted(csv_dir.glob("*.csv"))
print("Found:", csv_paths)

out_path = plot_dimer_scan(csv_paths, show=True)
print("Saved:", out_path)

for p in csv_paths:
    result = check_spurious_minimum(p)
    print(f"\n{p.stem}")
    for k, v in result.items():
        print(f"  {k}: {v}")

### B6. Download results

In [ ]:
from google.colab import files
import shutil

archive_path = shutil.make_archive("/content/test3_dimer_results_partB", "zip", str(RESULTS_DIR))
files.download(archive_path)